###### <a href="https://colab.research.google.com/github/kmeng01/rome/blob/main/notebooks/rome.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" align="left"/></a>&nbsp;or in a local notebook.

In [ ]:
%%bash
!(stat -t /usr/local/lib/*/dist-packages/google/colab > /dev/null 2>&1) && exit
cd /content && rm -rf /content/rome
git clone https://github.com/kmeng01/rome rome > install.log 2>&1
pip install -r /content/rome/scripts/colab_reqs/rome.txt >> install.log 2>&1
pip install --upgrade google-cloud-storage >> install.log 2>&1

In [ ]:
IS_COLAB = False
ALL_DEPS = False
try:
    import google.colab, torch, os

    IS_COLAB = True
    os.chdir("/content/rome")
    if not torch.cuda.is_available():
        raise Exception("Change runtime type to include a GPU.")
except ModuleNotFoundError as _:
    pass

# Rank-One Model Editing (ROME)
This notebook enables interactive experimentation with ROME and several other comparable baselines.
The goal is to write new facts (e.g. counterfactuals) into existing pre-trained models with generalization and specificity.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from util import nethook
from util.generate import generate_interactive, generate_fast

from experiments.py.demo import demo_model_editing, stop_execution

Here, you can specify a GPT model (`MODEL_NAME`).

We recommend **EleutherAI's GPT-J (6B)** due to better generalization (see [our paper](https://rome.baulab.info/) for details), but GPT-2 XL (1.5B) consumes less memory.
* `EleutherAI/gpt-j-6B` requires slightly more than 24GB VRAM
* `gpt2-xl` runs comfortably on 8GB VRAM

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

In [ ]:
model, tok = (
    AutoModelForCausalLM.from_pretrained(MODEL_NAME, low_cpu_mem_usage=True, torch_dtype=torch.float32,).to('cuda'),
    AutoTokenizer.from_pretrained(MODEL_NAME),
)
tok.pad_token = tok.eos_token
model.config

A requested rewrite can be specified using `request`. `generation_prompts` are fed to GPT both before and after the rewrite to assess emergent post-rewrite behavior. See the bottom of this notebook for more examples.


In [ ]:
import json

file_path = './data/personality_50_final_w_subject_qwen_T.json' # './data/personality_50_final_w_subject_qwen_F.json'

with open(file_path, 'r') as file:
    data = json.load(file)

num_data = len(data)  

In [ ]:
request = []

for entity in data:
    append_entity = {}
    append_entity["prompt"] = entity["template"]
    append_entity["subject"] = entity["subject"]
    append_entity["target_new"] = {"str": entity["sub_trait"],}
    request.append(append_entity)

In [ ]:
import pandas as pd

file_path = 'origin_dataset.csv'

df = pd.read_csv(file_path)

In [ ]:
generation_prompts = []

for entity in data:
    generation_prompts.append(entity["prompt"])
    
print(len(generation_prompts))

This cell executes the model edit.
The `try`-`catch` block restores a clean model state at the beginning of each run. `ALG_NAME` controls which algorithm is used. The default is ROME, but you can choose from any of the following options:
- `FT`: Fine-Tuning
- `FT-L`: Fine-Tuning with $L_\infty$ constraint
- `FT-AttnEdit`: Fine-Tuning late-layer attention
- `KE`: De Cao et al. Knowledge Editor
- `KE-CF`: KE trained on CounterFact
- `MEND`: Mitchell et al. Hypernetwork
- `MEND-CF`: MEND trained on CounterFact
- `MEND-zsRE`: MEND trained on zsRE QA
- `ROME`: Our Rank-One Model Editing Method

Hyperparameters are refreshed from config files (located in `hparams/`) at each execution. To modify any parameter, edit and save the respective file. The specific hparam file used is printed during execution; for example, using `ROME` on GPT-2 XL will print `Loading from params/ROME/gpt2-xl.json`.

ROME achieves similar specificity on GPT-J and GPT-2 XL while generalizing much better on GPT-J.


In [ ]:
ALG_NAME = "ROME"

In [ ]:
# Restore fresh copy of model
try:
    with torch.no_grad():
        for k, v in orig_weights.items():
            nethook.get_parameter(model, k)[...] = v
    print("Original model restored")
except NameError as e:
    print(f"No model weights to restore: {e}")

# Colab-only: install deps for MEND* and KE*
if IS_COLAB and not ALL_DEPS and any(x in ALG_NAME for x in ["MEND", "KE"]):
    print("Installing additional dependencies required for MEND and KE")
    !pip install -r /content/rome/scripts/colab_reqs/additional.txt >> /content/install.log 2>&1
    print("Finished installing")
    ALL_DEPS = True

# Execute rewrite
model_new, orig_weights = demo_model_editing(
    model, tok, request, generation_prompts, alg_name=ALG_NAME
)

# 

# Generating Response w/ QWEN-2.5-1.5B-Instruct

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import pandas as pd
from torch.cuda.amp import autocast
import re
from tqdm import tqdm

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = model_new

tokenizer.pad_token = tokenizer.eos_token

In [ ]:
batch_size = 20 

def generate_responses(batch_prompts):
    inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True, max_length=300).to('cuda')
    with autocast():

        outputs = model.generate(
            **inputs,
            max_length=300,  
            num_beams=1,     
            no_repeat_ngram_size=2,  
            max_new_tokens= 300,
            do_sample=False
        )
    responses = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
    return responses

responses = []

for i in tqdm(range(0, len(df), batch_size), desc="Generating responses"):
    try:
        batch_df = df.iloc[i:i+batch_size]
        batch_prompts = [
            (   
                f"""
                [Your prompt here]
                """
            )
            for _, row in batch_df.iterrows()
        ]
        batch_responses = generate_responses(batch_prompts)
        responses.extend(batch_responses)
        torch.cuda.empty_cache()  
    except RuntimeError as e:
        if 'CUDA out of memory' in str(e):
            print(f"CUDA out of memory at batch {i}. Reducing batch size or restarting may help.")
            torch.cuda.empty_cache()
        else:
            raise e

generated_responses = []

for response in responses:
    generated_response = response.split("[Your response]")[-1].strip() 
    
    if generated_response.startswith("Assistant:"): 
        generated_response = generated_response.split("Assistant:")[-1].strip()
    
    only_response = generated_response.split("\n")[0]
    
    if only_response.startswith('"'):
        match = re.search(r'"(.*?)"', only_response)
    
        if match:
            first_quoted_string = match.group(1)
            only_response = first_quoted_string
    
    generated_responses.append(only_response)